# Story 3.6 — Evaluation Rigor → Champion Selection

**Epic 3 capstone.** Epic 3 built five lenses for looking at a model:

1. **Discrimination** (3.1) — AUC-PR, precision@k
2. **Calibration** (3.2) — Brier, ECE: are the probabilities trustworthy?
3. **Threshold** (3.3) — where do we draw the flag/no-flag line?
4. **Expected value** (3.4) — what is the operating point worth in dollars?
5. **Unbiased generalisation** (3.5) — nested CV: an honest AUC-PR estimate

This notebook converges them into one decision: **of the six model × cohort cells (LR / GBM / EBM × `hris_only` / `hybrid`), which one do we ship?**

The rule is a **gated rank, not a sort** (`src/retention/evaluation/champion.py`): a cell is *eligible* only if its Brier clears the base-rate bar `β·(1 − β)` — the Brier of the best constant predictor — and the champion is the highest-AUC-PR cell among the eligible ones. A cell can rank well while being mis-calibrated, and every downstream HR action (the threshold, the EV case) is computed from probabilities, so ranking alone is not enough to operate on.

> **Test-set discipline.** Every selection number below is a **validation** number. The held-out test set is touched exactly once, in §11, to confirm the champion — never to choose it.

## 1 — Setup

In [ ]:
from __future__ import annotations

import datetime as dt
import warnings

import matplotlib.pyplot as plt
import pandas as pd

from retention import config
from retention.data.load import load_attrition_features_local
from retention.data.split import temporal_split
from retention.evaluation.calibration import expected_calibration_error
from retention.evaluation.champion import (
    ChampionArtifact,
    persist_champion,
    select_champion,
)
from retention.evaluation.expected_value import (
    DEFAULT_INTERVENTION_COST,
    DEFAULT_P_EFF,
    DEFAULT_REPLACEMENT_COST,
    breakeven_p_eff,
    ev_at_threshold,
    expected_value_plot,
    p_eff_sensitivity_sweep,
)
from retention.evaluation.metrics import auc_pr, brier_score, precision_at_k
from retention.evaluation.nested_cv import nested_cv_auc_pr
from retention.features.catalog import get_label_name
from retention.features.cohorts import extract_X_y, get_cohort_feature_names, split_cohorts
from retention.models.ebm import train_ebm
from retention.models.lr import train_lr
from retention.models.threshold import optimize_threshold, threshold_sweep, threshold_sweep_plot
from retention.models.tracking import register_champion
from retention.models.xgb import RetentionModel

# Notebook noise (sklearn/mlflow/xgboost deprecation chatter) hidden so the
# narrative output stays readable. The library code is fully typed and tested.
warnings.filterwarnings("ignore")
config.configure_plot_style()
config.set_global_seed()

LABEL = get_label_name()
CRITERION = "f2"  # [FLIP-RISK]: recall weighted 2x precision — a missed exit
# costs more than a wasted retention conversation, so the operating point sits
# lower (flags more people). Same criterion the threshold story (3.3) argues for.
print(f"SEED={config.SEED}  label={LABEL}  operating-point criterion={CRITERION}")

## 2 — Data & temporal split

We load from the committed **CSV snapshot** (the reviewer path: no BigQuery credentials needed) and split **temporally** — oldest 70 % train, next 15 % validation, newest 15 % test. No shuffling: HR data trends over time, and a random split would leak the future into the past.

The snapshot is pinned to `2026-05-28` (the date the methodology's documented nested-CV and EV numbers were computed on) so this notebook reproduces them, with a fallback to a later snapshot for a fresh clone.

In [ ]:
# config.DATA_DIR is absolute (resolved from the installed package), so the
# load is independent of the working directory — nbconvert runs with cwd=notebooks/.
data_csv = config.DATA_DIR / "raw" / "v_attrition_features_2026-05-28.csv"
if not data_csv.exists():
    data_csv = config.DATA_DIR / "raw" / "v_attrition_features_2026-05-29.csv"

df = load_attrition_features_local(data_csv)
print(f"Loaded {data_csv.name}: {len(df):,} rows x {df.shape[1]} cols")

config.set_global_seed()
train_df, val_df, test_df = temporal_split(df, save_indices=False)
print(
    f"train={len(train_df):,}  val={len(val_df):,}  test={len(test_df):,}  "
    f"(test held out until the single confirmation in section 11)"
)

val_base_rate = float(val_df[LABEL].mean())
print(
    f"base rates  ->  train={train_df[LABEL].mean():.3f}  "
    f"val={val_base_rate:.3f}  test={test_df[LABEL].mean():.3f}"
)

## 3 — Cohort splits

The controlled experiment at the heart of Loop 2: **identical rows, different columns.** `hris_only` sees only HRIS features; `hybrid` adds the three survey-signal columns. Any AUC-PR difference is attributable to the survey signal, not to a different employee population.

In [ ]:
train_cohorts = split_cohorts(train_df)
val_cohorts = split_cohorts(val_df)
test_cohorts = split_cohorts(test_df)

for cohort_name in ("hris_only", "hybrid"):
    n_feat = len(get_cohort_feature_names(cohort_name))
    print(f"  {cohort_name}: {n_feat} features")

## 4 — Train the six model × cohort cells

Each model is fitted on the **training** split only. Preprocessors are fit on train (no val/test leakage) and assembled into fitted Pipelines so `predict_proba(X_raw)` transforms automatically. LR/GBM share the one-hot preprocessor; EBM uses native categorical handling — an acknowledged tradeoff documented in `docs/methodology.md`.

In [ ]:
config.set_global_seed()
models = {}
for cohort in ("hris_only", "hybrid"):
    X_tr, y_tr = extract_X_y(train_cohorts[cohort], cohort)

    models[("LR", cohort)] = train_lr(X_tr, y_tr, cohort=cohort)

    gbm = RetentionModel(cohort=cohort)
    gbm.fit(X_tr, y_tr)
    models[("GBM", cohort)] = gbm

    models[("EBM", cohort)] = train_ebm(X_tr, y_tr, cohort=cohort)

    print(f"[{cohort}] LR, GBM, EBM trained")

## 5 — Cache the validation probabilities

Every lens below reads from the *same* frozen validation probabilities, so the summary table, the threshold sweep, and the EV case all describe one model — never differently-scored variants of it.

In [ ]:
val_proba = {}
for (model_name, cohort), model in models.items():
    X_val, _ = extract_X_y(val_cohorts[cohort], cohort)
    val_proba[(model_name, cohort)] = model.predict_proba(X_val)

print(f"Cached validation probabilities for {len(val_proba)} cells")

## 6 — Lens 1: Nested cross-validation (the optimism check)

A single validation score is optimistic: the model that wins on validation was, in part, *chosen* on validation. **Nested CV** separates the two jobs — an outer 5-fold loop measures performance on folds never seen during the inner 5-fold hyperparameter search — giving an unbiased AUC-PR estimate.

Run on the **development set (train + val)** with the test set withheld, and on **GBM only**: it has the largest hyperparameter surface and is the one most prone to small-data overfitting, so it is the cell whose public AUC-PR claim most needs the safeguard (LR/EBM rationale in `nested_cv.py`). The gap between the flat validation score and the nested mean *is* the optimism we are correcting for.

In [ ]:
# Nested CV is ~200 GBM fits per cohort — the slowest cell in the notebook
# (well under the raised nbconvert timeout). The test set is never passed in.
dev_df = pd.concat([train_df, val_df], ignore_index=True)
dev_cohorts = split_cohorts(dev_df)

nested_results = {}
for cohort in ("hris_only", "hybrid"):
    X_dev, y_dev = extract_X_y(dev_cohorts[cohort], cohort)
    config.set_global_seed()
    nested_results[cohort] = nested_cv_auc_pr(X_dev, y_dev, cohort=cohort)
    print(nested_results[cohort].summary())

## 7 — The six-cell summary table

One row per model × cohort, all on the **validation** set:

- **auc_pr** — discrimination (primary metric)
- **prec_at_10** — of the top 10 % flagged, how many actually exit
- **ece** — calibration error (lower = predicted probs match reality)
- **brier** — squared-error calibration score (drives the gate)
- **threshold** — the cell's own F2-optimal operating point
- **ev_at_default** — EV in dollars at that threshold, p_eff = 0.30
- **breakeven_p_eff** — intervention effectiveness at which EV = 0

This table is the input to `select_champion`.

In [ ]:
rows = []
for (model_name, cohort), proba in val_proba.items():
    _, y_val = extract_X_y(val_cohorts[cohort], cohort)
    threshold = optimize_threshold(y_val, proba, CRITERION)
    rows.append(
        {
            "model": model_name,
            "cohort": cohort,
            "auc_pr": round(auc_pr(y_val, proba), 3),
            "prec_at_10": round(precision_at_k(y_val, proba, k=0.10), 3),
            "ece": round(expected_calibration_error(y_val, proba), 3),
            "brier": round(brier_score(y_val, proba), 3),
            "threshold": round(threshold, 2),
            "ev_at_default": round(ev_at_threshold(y_val, proba, threshold), 0),
            "breakeven_p_eff": round(breakeven_p_eff(y_val, proba, threshold), 3),
        }
    )

summary = pd.DataFrame(rows)
summary

**Flat vs nested — the optimism gap.** Side by side, the single validation AUC-PR for GBM against its unbiased nested-CV mean. A flat score above the nested mean is the over-optimism nested CV exists to expose; the nested figure is the one to quote publicly.

In [ ]:
for cohort in ("hris_only", "hybrid"):
    flat = summary.loc[
        (summary["model"] == "GBM") & (summary["cohort"] == cohort), "auc_pr"
    ].iloc[0]
    nested = nested_results[cohort]
    gap = flat - nested.mean
    print(
        f"GBM x {cohort:9s}:  flat val AUC-PR={flat:.3f}   "
        f"nested {nested.outer_splits}x{nested.inner_splits} CV={nested.mean:.3f} "
        f"+/- {nested.std:.3f}   (optimism gap {gap:+.3f})"
    )

## 8 — Champion selection: calibration gate, then AUC-PR rank

`select_champion` applies the rule:

1. **Gate** — keep only cells with `brier ≤ β·(1 − β)`, the base-rate Brier at the validation prevalence β. A cell above that bar is worse than predicting the base rate for everyone — its probabilities are not safe to operate, no matter how it ranks.
2. **Rank** — among the eligible cells, take the highest AUC-PR.
3. **Honest fallback** — if *no* cell clears the gate, pick the highest-AUC-PR cell overall, flag `passed_calibration_gate = False`, and say so: the ranking is usable, the probabilities need calibration first.

The printed summary reports which path was taken — it is not assumed in advance.

In [ ]:
selection = select_champion(summary, base_rate=val_base_rate, criterion=CRITERION)
print(selection.summary())

## 9 — The champion's operating point

Threshold selection is a *within-system A/B test*: the model's ranking is frozen, and each candidate threshold is a treatment arm ("flag everyone with p ≥ t") scored on the same validation set. The dashed line marks the F2-optimal arm — the [FLIP-RISK] operating point the champion will ship with.

In [ ]:
champ_key = (selection.model_name, selection.cohort)
champion_model = models[champ_key]
proba_val_champ = val_proba[champ_key]
_, y_val_champ = extract_X_y(val_cohorts[selection.cohort], selection.cohort)

sweep = threshold_sweep(y_val_champ, proba_val_champ)
threshold_sweep_plot(
    sweep,
    optimal_threshold=selection.threshold,
    criterion=CRITERION,
    model_name=selection.model_name,
    cohort=selection.cohort,
)
plt.show()

## 10 — The business case: expected value & breakeven

An F-score is unit-free — it never prices a false positive. This puts the operating point in **dollars**: 

$$EV = p_{eff} \cdot \text{replacement\_cost} \cdot TP - \text{intervention\_cost} \cdot (TP + FP)$$

`p_eff` (how often a retention conversation actually works) is a forward assumption we cannot read off the data, so we never report a single number — we **sweep** it across [0.1, 0.9] and show the **breakeven**: the effectiveness at which the program starts paying for itself. The dataset carries a salary *ratio*, not absolute salary, so `replacement_cost` uses the documented default (1.5 × $60k = $90k, SHRM 2024).

In [ ]:
ev_sweep = p_eff_sensitivity_sweep(y_val_champ, proba_val_champ, selection.threshold)
be = breakeven_p_eff(y_val_champ, proba_val_champ, selection.threshold)
ev_central = ev_at_threshold(y_val_champ, proba_val_champ, selection.threshold)

print(
    f"replacement_cost=${DEFAULT_REPLACEMENT_COST:,.0f}   "
    f"intervention_cost=${DEFAULT_INTERVENTION_COST:,.0f}"
)
print(f"breakeven p_eff = {be:.3f}    EV @ p_eff={DEFAULT_P_EFF:.2f} = ${ev_central:,.0f}")

expected_value_plot(
    ev_sweep,
    breakeven=be,
    threshold=selection.threshold,
    model_name=selection.model_name,
    cohort=selection.cohort,
)
plt.show()

## 11 — Test-set confirmation (once)

The **only** time the held-out test set is touched. The champion was chosen entirely on validation; here we confirm it generalises. Both validation and test metrics are stored on the artifact so the optimism gap is visible in the persisted record, not hidden.

In [ ]:
X_test_champ, y_test_champ = extract_X_y(test_cohorts[selection.cohort], selection.cohort)
proba_test_champ = champion_model.predict_proba(X_test_champ)

val_metrics = {
    "auc_pr": float(auc_pr(y_val_champ, proba_val_champ)),
    "prec_at_10": float(precision_at_k(y_val_champ, proba_val_champ, k=0.10)),
    "ece": float(expected_calibration_error(y_val_champ, proba_val_champ)),
    "brier": float(brier_score(y_val_champ, proba_val_champ)),
}
test_metrics = {
    "auc_pr": float(auc_pr(y_test_champ, proba_test_champ)),
    "prec_at_10": float(precision_at_k(y_test_champ, proba_test_champ, k=0.10)),
    "ece": float(expected_calibration_error(y_test_champ, proba_test_champ)),
    "brier": float(brier_score(y_test_champ, proba_test_champ)),
}

gap = pd.DataFrame({"validation": val_metrics, "test": test_metrics})
gap["optimism_gap"] = gap["validation"] - gap["test"]
gap.round(3)

## 12 — Persist the champion artifact

`ChampionArtifact` bundles the fitted model + its operating threshold + full provenance (the selection decision, val/test metrics, seed, timestamp) and pickles it to `reports/models/champion.pkl` — the self-describing record Loop 4's write-back will load.

In [ ]:
artifact = ChampionArtifact(
    model=champion_model,
    model_name=selection.model_name,
    cohort=selection.cohort,
    threshold=selection.threshold,
    feature_names=get_cohort_feature_names(selection.cohort),
    seed=config.SEED,
    selection=selection,
    val_metrics=val_metrics,
    test_metrics=test_metrics,
    created_utc=dt.datetime.now(dt.timezone.utc).isoformat(),
)
champion_path = persist_champion(artifact)
print(f"persisted -> {champion_path}\n")
print(artifact.summary())

## 13 — Register to the MLflow Model Registry → Production

**Story 2.7.10.** Where the experiment view records *every* run, the registry records the *one* winner: a named, versioned, stage-tagged entry. `register_champion` logs the fitted estimator, registers it under `rp-champion`, and promotes the new version to **Production** — the URI Loop 4 loads from (`models:/rp-champion/Production`). We pass the sklearn object (`RetentionModel.pipeline` for GBM; LR/EBM are Pipelines already) because `mlflow.sklearn` needs the estimator, not the wrapper.

In [ ]:
# RetentionModel exposes the fitted sklearn Pipeline as .pipeline; LR/EBM
# champions already are Pipelines. getattr handles both.
sk_model = getattr(champion_model, "pipeline", champion_model)

# Explicit pip_requirements per family skips MLflow's slow (~20s) environment
# inference on every call — fine here because we know each family's deps.
pip_by_family = {
    "GBM": ["scikit-learn", "xgboost"],
    "LR": ["scikit-learn"],
    "EBM": ["scikit-learn", "interpret"],
}

run_id, version = register_champion(
    sk_model,
    params={
        "model": selection.model_name,
        "cohort": selection.cohort,
        "seed": config.SEED,
        "threshold": selection.threshold,
        "criterion": selection.criterion,
        "passed_calibration_gate": selection.passed_calibration_gate,
    },
    metrics={f"test_{k}": v for k, v in test_metrics.items()},
    pip_requirements=pip_by_family.get(selection.model_name, ["scikit-learn"]),
)
print(f"registered rp-champion v{version} (run {run_id[:8]}) -> Production")

## 14 — Summary

The five Epic 3 lenses converged into one gated-rank decision. The champion is selected on validation, confirmed once on test, persisted to `reports/models/champion.pkl`, and registered as `rp-champion/Production`.

**Reproduce:** `make evaluate` regenerates and re-executes this notebook end to end. **Inspect the registry:** `make mlflow-ui`, then the *Models* tab → `rp-champion` → the *Production* version.

Loop 4 loads the champion with `retention.evaluation.champion.load_champion()` (file store) or `mlflow.sklearn.load_model('models:/rp-champion/Production')` (registry) — same model, two access paths.